In [36]:
import os
os.chdir(r'D:\8th semester\Machine Learning Lab\par_data_set')

In [37]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [38]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [39]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [40]:
model1 = MLP()
model1.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_4 (Flatten)         (None, 504)               0         
                                                                 
 dense_8 (Dense)             (None, 32)                16160     
                                                                 
 dense_9 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16193 (63.25 KB)
Trainable params: 16193 (63.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [41]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [42]:
checkpoints = r'D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\8th semester\Machine Learning Lab\par_data_set'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [43]:
os.path.exists(JSON_PATH)

False

In [44]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [45]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [46]:
import os
path_dataset =r'D:\8th semester\Machine Learning Lab\par_data_set'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [47]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.0037932395935058594 sec


In [48]:
train_X.shape

(835, 24, 21)

In [49]:
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/5
 1/27 [>.............................] - ETA: 9s - loss: 0.5762 - mae: 0.5762 - mape: 196.4375
Epoch 1: val_loss improved from inf to 0.09596, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0001-loss0.10.h5
27/27 [==============================] - 1s 9ms/step - loss: 0.1845 - mae: 0.1845 - mape: 98.7328 - val_loss: 0.0960 - val_mae: 0.0960 - val_mape: 30.2269
Epoch 2/5
27/27 [==============================] - ETA: 0s - loss: 0.1062 - mae: 0.1062 - mape: 50.0060
Epoch 2: val_loss did not improve from 0.09596


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 13ms/step - loss: 0.1062 - mae: 0.1062 - mape: 50.0060 - val_loss: 0.1306 - val_mae: 0.1306 - val_mape: 47.9837
Epoch 3/5
 1/27 [>.............................] - ETA: 0s - loss: 0.0720 - mae: 0.0720 - mape: 29.2605
Epoch 3: val_loss improved from 0.09596 to 0.08041, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0003-loss0.08.h5
27/27 [==============================] - 0s 7ms/step - loss: 0.0824 - mae: 0.0824 - mape: 42.2347 - val_loss: 0.0804 - val_mae: 0.0804 - val_mape: 27.8486
Epoch 4/5
 1/27 [>.............................] - ETA: 0s - loss: 0.1021 - mae: 0.1021 - mape: 67.8195
Epoch 4: val_loss did not improve from 0.08041


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 6ms/step - loss: 0.0809 - mae: 0.0809 - mape: 41.4612 - val_loss: 0.0949 - val_mae: 0.0949 - val_mape: 34.0030
Epoch 5/5
 1/27 [>.............................] - ETA: 0s - loss: 0.1472 - mae: 0.1472 - mape: 63.6529
Epoch 5: val_loss improved from 0.08041 to 0.07014, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5
27/27 [==============================] - 0s 7ms/step - loss: 0.0855 - mae: 0.0855 - mape: 46.1310 - val_loss: 0.0701 - val_mae: 0.0701 - val_mape: 19.6382


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [51]:

model = load_model(r'D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 125ms/step
Mean Absolute Error (MAE): 9319.53
Median Absolute Error (MedAE): 9573.4
Mean Squared Error (MSE): 88696856.14
Root Mean Squared Error (RMSE): 9417.9
Mean Absolute Percentage Error (MAPE): 59.66 %
Median Absolute Percentage Error (MDAPE): 61.86 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [52]:
checkpoints = r'D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5'
model=r'D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5'
start_epoch= 7

In [53]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [54]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
 1/27 [>.............................] - ETA: 6s - loss: 0.0753 - mae: 0.0753 - mape: 30.9256
Epoch 1: val_loss improved from inf to 0.07498, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5
27/27 [==============================] - 1s 12ms/step - loss: 0.0566 - mae: 0.0566 - mape: 27.5097 - val_loss: 0.0750 - val_mae: 0.0750 - val_mape: 23.7150
Epoch 2/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0447 - mae: 0.0447 - mape: 44.6904

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



Epoch 2: val_loss improved from 0.07498 to 0.06360, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 14ms/step - loss: 0.0503 - mae: 0.0503 - mape: 26.6945 - val_loss: 0.0636 - val_mae: 0.0636 - val_mape: 19.2958
Epoch 3/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0421 - mae: 0.0421 - mape: 16.3101
Epoch 3: val_loss improved from 0.06360 to 0.05720, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5


...

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Epoch 5/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0476 - mae: 0.0476 - mape: 24.6102
Epoch 5: val_loss did not improve from 0.05680
27/27 [==============================] - 0s 10ms/step - loss: 0.0470 - mae: 0.0470 - mape: 23.0969 - val_loss: 0.0669 - val_mae: 0.0669 - val_mape: 20.7677
Epoch 6/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0537 - mae: 0.0537 - mape: 24.2570
Epoch 6: val_loss did not improve from 0.05680
27/27 [==============================] - 0s 6ms/step - loss: 0.0464 - mae: 0.0464 - mape: 23.3137 - val_loss: 0.0879 - val_mae: 0.0879 - val_mape: 28.7740
Epoch 7/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0520 - mae: 0.0520 - mape: 20.9264
Epoch 7: val_loss did not improve from 0.05680
27/27 [==============================] - 0s 6ms/step - loss: 0.0472 - mae: 0.0472 - mape: 25.1885 - val_loss: 0.0682 - val_mae: 0.0682 - val_mape: 21.7023
Epoch 8/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0494 - m

c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



Epoch 9: val_loss did not improve from 0.05122
27/27 [==============================] - 0s 7ms/step - loss: 0.0444 - mae: 0.0444 - mape: 23.0595 - val_loss: 0.0526 - val_mae: 0.0526 - val_mape: 16.0019
Epoch 10/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0450 - mae: 0.0450 - mape: 12.8812
Epoch 10: val_loss improved from 0.05122 to 0.05022, saving model to D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5
27/27 [==============================] - 0s 10ms/step - loss: 0.0438 - mae: 0.0438 - mape: 22.0611 - val_loss: 0.0502 - val_mae: 0.0502 - val_mape: 14.8281


c:\Users\Zohaib\anaconda3\envs\XAI\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [55]:

model = load_model(r'D:\8th semester\Machine Learning Lab\par_data_set\E1-cp-0005-loss0.07.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 44ms/step
Mean Absolute Error (MAE): 9939.74
Median Absolute Error (MedAE): 10018.19
Mean Squared Error (MSE): 100080739.45
Root Mean Squared Error (RMSE): 10004.04
Mean Absolute Percentage Error (MAPE): 63.6 %
Median Absolute Percentage Error (MDAPE): 64.73 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
